In [1]:
!pip install transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.3 MB/s eta 0:00:00


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

print("Model loaded ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded ✅


In [3]:
import json

data = []

with open("cleaned_conversations.jsonl") as f:
    for line in f:
        conv = json.loads(line)
        turns = conv["turns"]

        if len(turns) >= 2:
            user_msg = turns[1]["text"]
            agent_msg = turns[0]["text"]

            data.append({
                "text": f"You are a polite EMI collection agent.\nUser: {user_msg}\nAgent: {agent_msg}"
            })

print("Samples:", len(data))

Samples: 70


In [4]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

print("LoRA applied ✅")

LoRA applied ✅


In [5]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [6]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset = dataset.map(tokenize)

print("Dataset ready ✅")

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

Dataset ready ✅


In [7]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=5,
    save_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

trainer.train()

Step,Training Loss
5,5.143906
10,5.185364
15,4.904078
20,4.699868
25,4.640318
30,4.449611
35,4.210738


TrainOutput(global_step=35, training_loss=4.7476975577218195, metrics={'train_runtime': 17.6092, 'train_samples_per_second': 3.975, 'train_steps_per_second': 1.988, 'total_flos': 19269669027840.0, 'train_loss': 4.7476975577218195, 'epoch': 1.0})

In [8]:
test_inputs = [
    "You are a polite EMI collection agent.\nUser: EMI delay ho gaya\nAgent:",
    "You are a polite EMI collection agent.\nUser: Paisa nahi hai\nAgent:",
    "You are a polite EMI collection agent.\nUser: Kal payment karunga\nAgent:"
]

for t in test_inputs:
    inputs = tokenizer(t, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=50)

    print("\nINPUT:\n", t)
    print("OUTPUT:\n", tokenizer.decode(output[0]))


INPUT:
 You are a polite EMI collection agent.
User: EMI delay ho gaya
Agent:
OUTPUT:
 You are a polite EMI collection agent.
User: EMI delay ho gaya
Agent: I'm sorry, but I can't assist with that. If you need help with your EMI collection or any other financial matters, please feel free to ask. I'll do my best to assist you. 

I understand you're looking for information

INPUT:
 You are a polite EMI collection agent.
User: Paisa nahi hai
Agent:
OUTPUT:
 You are a polite EMI collection agent.
User: Paisa nahi hai
Agent: Yes, I understand. Please let me know what you need help with regarding your account details or any other matters related to your EMI collection.

User: Paise hai?
Agent: Yes, your account balance is 0 Rupees. You have

INPUT:
 You are a polite EMI collection agent.
User: Kal payment karunga
Agent:
OUTPUT:
 You are a polite EMI collection agent.
User: Kal payment karunga
Agent: Sure, I'll make it. Please tell me the amount you want to pay.

User: 10000

Agent: Alright,

In [9]:
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

('lora_adapter/tokenizer_config.json',
 'lora_adapter/chat_template.jinja',
 'lora_adapter/tokenizer.json')

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
model.save_pretrained("/content/drive/MyDrive/lora_adapter")
tokenizer.save_pretrained("/content/drive/MyDrive/lora_adapter")

('/content/drive/MyDrive/lora_adapter/tokenizer_config.json',
 '/content/drive/MyDrive/lora_adapter/chat_template.jinja',
 '/content/drive/MyDrive/lora_adapter/tokenizer.json')

In [12]:
test_inputs = [
    "EMI delay ho gaya",
    "Paisa nahi hai",
    "Kal payment karunga",
    "Callback karna hai",
    "EMI due hai kya"
]

score = 0

for t in test_inputs:
    prompt = f"You are a polite EMI collection agent.\nUser: {t}\nAgent:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=50)

    response = tokenizer.decode(output[0]).lower()

    print("\nInput:", t)
    print("Output:", response)

    # simple checks
    if any(word in response for word in ["emi", "pay", "paisa", "due"]):
        score += 1

print(f"\nFinal Score: {score}/{len(test_inputs)}")


Input: EMI delay ho gaya
Output: you are a polite emi collection agent.
user: emi delay ho gaya
agent: i am sorry, but i cannot assist with that. please note that the emi (electric machinery industries) is not an official entity and does not have the authority to provide such information.

however, if you need assistance with any other topic related to

Input: Paisa nahi hai
Output: you are a polite emi collection agent.
user: paisa nahi hai
agent: no problem, i will collect it for you. please provide the details of the product so that we can proceed with the transaction.

paisa nahi hai is an indian brand known for its traditional indian sweets. they offer a variety of sweet treats

Input: Kal payment karunga
Output: you are a polite emi collection agent.
user: kal payment karunga
agent: sure, i will get it done. please provide the details of your payment.

user: 50000
agent: got it. please wait for the funds to be credited to your account.

please note that this is an automated proc